In [0]:
from datetime import date

In [0]:
%pip install --upgrade numpy pandas pyarrow db-dtypes google-cloud-bigquery google-auth
dbutils.library.restartPython()

# 01 — Ingestão Batch Mensal

Ingestão mensal das tabelas de referência do **Indicador Criança Alfabetizada** (INEP / Base dos Dados) via BigQuery.

**Tabelas ingeridas:**

| DataFrame | Tabela BigQuery | Destino (`origens`) |
|---|---|---|
| `df_meta_alf_br` | `meta_alfabetizacao_brasil` | `origens.tc02_meta_brasil` |
| `df_meta_alf_uf` | `meta_alfabetizacao_uf` | `origens.tc02_meta_uf` |
| `df_meta_alf_mun` | `meta_alfabetizacao_municipio` | `origens.tc02_meta_mun` |
| `df_alunos` | `alunos` | `origens.tc02_alunos` |
| `df_municipio` | `municipio` | `origens.tc02_municipio` |
| `df_uf` | `uf` | `origens.tc02_uf` |

Cada execução grava na partição `(_ano_ingestao, _mes_ingestao)` do mês corrente, preservando o histórico de execuções anteriores.

**Próximo passo:** executar `02_carga_camada_silver.py`

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BatchIngestion") \
    .getOrCreate()


## 1. Credenciais BigQuery

O JSON completo da service account é lido do **Databricks Secrets** (scope `tc_02_gcp`, key `gcp_sa_json`),
eliminando qualquer arquivo de credencial no Workspace.
Mesmo padrão adotado pela ingestão streaming (`00_streaming_ingestion.ipynb`).

In [0]:
_sa_json_str = dbutils.secrets.get(scope="tc_02_gcp", key="gcp_sa_json")
chave = json.loads(_sa_json_str)

In [0]:
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_info(chave)
client = bigquery.Client(credentials=credentials, project="aist-tech-challenge02")


In [0]:
import base64

# _sa_json_str já é a string JSON exata do secret — reutilizada sem re-serialização
sa_b64 = base64.b64encode(_sa_json_str.encode("utf-8")).decode("utf-8")

In [0]:
def read_bq_table(table_name: str):
    return (
        spark.read.format("bigquery")
        .option("table", f"basedosdados.br_inep_avaliacao_alfabetizacao.{table_name}")
        .option("parentProject", "aist-tech-challenge02")
        .option("credentials", sa_b64)
        .load()
    )

## 2. Leitura das tabelas via BigQuery

In [0]:
print("Iniciando carregamento de meta_alfabetizacao_brasil:")
df_meta_alf_br = read_bq_table("meta_alfabetizacao_brasil")
print("Iniciando carregamento de meta_alfabetizacao_uf:")
df_meta_alf_uf = read_bq_table("meta_alfabetizacao_uf")
print("Iniciando carregamento de meta_alfabetizacao_municipio:")
df_meta_alf_mun = read_bq_table("meta_alfabetizacao_municipio")
print("Iniciando carregamento de alunos:")
df_alunos = read_bq_table("alunos")
print("Iniciando carregamento de municipio:")
df_municipio = read_bq_table("municipio")
print("Iniciando carregamento de dicionario:")
df_dicionario = read_bq_table("dicionario")
print("Iniciando carregamento de uf:")
df_uf = read_bq_table("uf")
print("\nTodas as tabelas carregadas.")

## 3. Persistência Mensal no Schema `origens`

Cada tabela recebe as colunas de controle `_data_ingestao`, `_ano_ingestao` e `_mes_ingestao`.
A escrita usa **Dynamic Partition Overwrite**: reprocessamentos no mesmo mês sobrescrevem apenas a partição corrente, sem afetar meses anteriores.

In [0]:
_hoje = date.today()
_ano = _hoje.year
_mes = _hoje.month

spark.sql("CREATE SCHEMA IF NOT EXISTS origens")

tabelas_batch = {
    "origens.tc02_meta_brasil": df_meta_alf_br,
    "origens.tc02_meta_uf":     df_meta_alf_uf,
    "origens.tc02_meta_mun":    df_meta_alf_mun,
    "origens.tc02_municipio":   df_municipio,
    "origens.tc02_alunos":      df_alunos,
    "origens.tc02_uf":          df_uf,
}

for nome_tabela, df in tabelas_batch.items():
    df_com_meta = (
        df
        .withColumn("_data_ingestao",  F.current_timestamp())
        .withColumn("_ano_ingestao",   F.lit(_ano).cast(T.ShortType()))
        .withColumn("_mes_ingestao",   F.lit(_mes).cast(T.ByteType()))
    )
    (
        df_com_meta.write
        .format("delta")
        .mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .option("mergeSchema", "true")
        .partitionBy("_ano_ingestao", "_mes_ingestao")
        .saveAsTable(nome_tabela)
    )
    contagem = spark.read.table(nome_tabela).count()
    print(f"  ✓ {nome_tabela}: {contagem} linhas")

print(f"\nIngestão mensal {_ano}-{_mes:02d} concluída com sucesso.")